In [ ]:
import numpy as np
from collections import Counter
import os

file = "p01_0.npy"
file_size = os.path.getsize(file)
print(f"File size of {file}: {file_size} bytes")
data = np.load(file)
print("Data", data)
print("Data size", data.shape)

def recurrence_matrix(state_space):
    M = state_space.shape[0]
    dist_matrix = np.linalg.norm(
        state_space[:, None, :] - state_space[None, :, :], axis=2
    )
    # print("Distance Matrix:\n", dist_matrix)
    epsilon = 0.1 * np.max(dist_matrix)
    # print("Epsilon:", epsilon)
    R = _heavside_fn(epsilon - dist_matrix)
    return R

def _heavside_fn(x):
    return np.where(x <= 0, 0, 1)

def state_space_reconstruction(data, m , tau):
    N = len(data)
    M = N - (m - 1) * tau
    state_space = np.zeros((M, m))
    for i in range(M):
        state_space[i] = data[i:i + m * tau:tau]
    return state_space

def _lengths_of_consecutive_ones(arr):
    array = np.asarray(arr, dtype=np.int8)
    if array.size == 0:
        return np.array([], dtype=int)
    edges = np.diff(np.r_[0, array, 0])
    starts = np.flatnonzero(edges == 1)
    ends = np.flatnonzero(edges == -1)
    return (ends - starts).tolist()

def p_of_l(Rb, M):
    counts = Counter()
    for k in range(-(M - 1), M):
        diag = np.diagonal(Rb, offset=k)
        # print(f"Offset {k} gives {list(diag)}")
        for L in _lengths_of_consecutive_ones(diag):
            counts[L] += 1
    return dict(counts)

def p_of_v(Rb, M):
    counts = {}
    for j in range(M):  # each column
        col = Rb[:, j]
        runs = _lengths_of_consecutive_ones(col)
        # print(f"Column {j} runs: {runs}")
        for v in runs:
            counts[v] = counts.get(v, 0) + 1
    return counts

def RQA(data, m , tau, l_min = 2, v_min= 2,  exclude_loi=True):
    state_space = state_space_reconstruction(data, m, tau)
    R = recurrence_matrix(state_space)
    M = R.shape[0]
    Rb = (np.asarray(R) != 0).astype(np.uint8)
    if exclude_loi:
        np.fill_diagonal(Rb, 0)
    R_sum = Rb.sum()
    RR = float(R_sum / Rb.size)
    counts = p_of_l(Rb, M)
    counts_vertical = p_of_v(Rb, M)
    counts_lmin = {l: c for l, c in counts.items() if l >= l_min}
    counts_vertical_lmin = {v: c for v, c in counts_vertical.items() if v >= v_min}
    total_counts = sum(counts_lmin.values())
    counts_vertical_sum = sum(counts_vertical_lmin.values())
    sum_l_pl = sum(L * c for L, c in counts_lmin.items())
    sum_l_pl_all = sum(L * c for L, c in counts.items())
    sum_v_pl = sum(v * c for v, c in counts_vertical_lmin.items())
    sum_v_pl_all = sum(v * c for v, c in counts_vertical.items())
    determinism = sum_l_pl / sum_l_pl_all if sum_l_pl_all else np.nan
    probs = {L: c / total_counts for L, c in counts_lmin.items()} if total_counts else {}
    entropy = float(-sum(p * np.log(p) for p in probs.values()) if probs else np.nan)
    L_max = max(counts_lmin.keys()) if counts_lmin else np.nan
    trapping_time = sum_v_pl / counts_vertical_sum if counts_vertical_sum else np.nan
    laminarity = sum_v_pl / sum_v_pl_all if sum_v_pl_all else np.nan

    return {
        "RR": RR,
        "determinism": determinism,
        "entropy": entropy,
        "L_max": L_max,
        "trapping_time": trapping_time,
        "laminarity": laminarity
    }


results = RQA(data, m=3, tau=3)
print(results)



File size of p01_0.npy: 30848 bytes
Data [-0.795  0.35  -0.73  ... -0.61   0.365 -0.78 ]
Data size (3840,)
{'RR': 0.12167183825144107, 'determinism': 0.7893277242326345, 'entropy': 2.7461949147092093, 'L_max': 95, 'trapping_time': 3.005791505791506, 'laminarity': 0.014799370653534035}


In [41]:
from tqdm import tqdm

folder = "/Users/weijithwimalasiri/Desktop/FYP/Datasets/AFPDB_Segments"
AF_folder = os.path.join(folder, "AF")
Pre_AF_folder = os.path.join(folder, "Pre-AF")
SR_folder = os.path.join(folder, "SR")

RR_values = []
determinism_values = []
entropy_values = []
L_max_values = []
trapping_time_values = []
laminarity_values = []

for filename in tqdm(os.listdir(SR_folder), desc="Processing SR files"):
    if filename.endswith(".npy"):
        file_path = os.path.join(SR_folder, filename)
        data = np.load(file_path)
        results = RQA(data, m=3, tau=3)
        RR_values.append(results["RR"])
        determinism_values.append(results["determinism"])
        entropy_values.append(results["entropy"])
        L_max_values.append(results["L_max"])
        trapping_time_values.append(results["trapping_time"])
        laminarity_values.append(results["laminarity"])


Processing SR files:   2%|▏         | 611/30749 [06:21<5:13:26,  1.60it/s]


KeyboardInterrupt: 